# Stg Layer — comment_sentiment (LLM, Incremental, added 2026-11-01)

Sentiment of YouTube comments on **food review videos that mention a restaurant** (`stg_restaurant_mentions`), from `${silver}.stg_comments` → `${silver}.stg_comment_sentiment` (1 row / comment). Feeds `gold/agg_restaurant_comments` and the dashboard page "Khán giả nói gì".

- `sentiment`: `positive` / `negative` / `neutral` about the food or the place (a question, a joke, a tag of a friend = `neutral`).
- `about_food`: the comment talks about the food, the price, the service or the place (not only about the YouTuber, the music or the video).
- **Incremental:** only comments not analyzed yet, or whose text changed (`text_checksum`, includes `PROMPT_VERSION`). Most liked first, at most `MAX_COMMENTS` per run (the first backfill of ~23k comments takes a few runs).
- Video skipped by the classifier (`is_food_review = false`) are not analyzed.

In [ ]:
# --- Config ---
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))
dbutils.widgets.text("max_comments", "5000")
MAX_COMMENTS = int(dbutils.widgets.get("max_comments"))

COMMENTS = f"{SILVER}.stg_comments"
MENTIONS = f"{SILVER}.stg_restaurant_mentions"
CLASSES = f"{SILVER}.stg_video_classification"
TARGET = f"{SILVER}.stg_comment_sentiment"
MODEL = "databricks-qwen3-next-80b-a3b-instruct"  # same model as video_classification / restaurant_mentions
MAX_TEXT_CHARS = 600

PROMPT = (
    "You read one Vietnamese YouTube comment under a food review video about an eating place. "
    "sentiment = the commenter's opinion of the food or the place: positive (praise, wants to go back, recommends), "
    "negative (complains about taste, price, hygiene, service, says it is an ad or overrated), "
    "neutral (question, tag of a friend, joke, no opinion). "
    "about_food = true when the comment is about the food, drinks, price, service or the place itself; "
    "false when it is only about the YouTuber, the video, music or something else."
)
RESPONSE_FORMAT = """{"type":"json_schema","json_schema":{"name":"comment_sentiment","schema":{"type":"object","properties":{"sentiment":{"type":"string","enum":["positive","negative","neutral"]},"about_food":{"type":"boolean"}},"required":["sentiment","about_food"]},"strict":true}}"""
RESULT_SCHEMA = "STRUCT<sentiment: STRING, about_food: BOOLEAN>"
import hashlib
PROMPT_VERSION = hashlib.md5((MODEL + PROMPT + RESPONSE_FORMAT).encode()).hexdigest()[:8]
print(f"Source: {COMMENTS} · target: {TARGET} · model {MODEL} · max {MAX_COMMENTS} comments / run")

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TARGET} (
        comment_id     STRING,
        video_id       STRING,
        sentiment      STRING,
        about_food     BOOLEAN,
        text_checksum  STRING,
        raw_response   STRING,
        model_name     STRING,
        analyzed_at    TIMESTAMP
    )
    USING DELTA
    COMMENT 'LLM sentiment of comments on food review videos that mention a restaurant (1 row per comment).'
""")

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW comments_todo AS
    WITH food_videos AS (
        SELECT DISTINCT m.video_id
        FROM {MENTIONS} m
        LEFT JOIN {CLASSES} c ON c.video_id = m.video_id
        WHERE COALESCE(c.is_food_review, true)
    ),
    src AS (
        SELECT c.comment_id, c.video_id, c.text, c.like_count,
               md5(concat(coalesce(c.text, ''), '{PROMPT_VERSION}')) AS text_checksum
        FROM {COMMENTS} c JOIN food_videos v ON v.video_id = c.video_id
        WHERE length(trim(coalesce(c.text, ''))) > 1
    )
    SELECT s.* FROM src s
    LEFT JOIN {TARGET} t ON t.comment_id = s.comment_id
    WHERE t.comment_id IS NULL OR t.text_checksum <> s.text_checksum
    ORDER BY s.like_count DESC NULLS LAST, s.comment_id
    LIMIT {MAX_COMMENTS}
""")
todo = spark.table("comments_todo").count()
print(f"Comments to analyze this run: {todo}")

In [ ]:
if todo > 0:
    prompt_sql = PROMPT.replace("'", "''")
    spark.sql(f"""
        MERGE INTO {TARGET} AS t
        USING (
            SELECT comment_id, video_id, r.sentiment, r.about_food, text_checksum, raw_response,
                   '{MODEL}' AS model_name, current_timestamp() AS analyzed_at
            FROM (
                SELECT *, from_json(raw_response, '{RESULT_SCHEMA}') AS r
                FROM (
                    SELECT comment_id, video_id, text_checksum,
                           ai_query('{MODEL}', concat('{prompt_sql}', '\\n\\nCOMMENT: ', left(text, {MAX_TEXT_CHARS})),
                                    responseFormat => '{RESPONSE_FORMAT}') AS raw_response
                    FROM comments_todo
                )
            )
        ) AS s
        ON t.comment_id = s.comment_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"✅ Analyzed {todo} comment(s)")
else:
    print("Nothing new to analyze ✅")

s = spark.sql(f"""
    SELECT COUNT(*) AS n, COUNT(DISTINCT comment_id) AS grain, COUNT_IF(sentiment IS NULL) AS unparsed,
           COUNT_IF(sentiment = 'positive') AS pos, COUNT_IF(sentiment = 'negative') AS neg,
           COUNT_IF(sentiment = 'neutral') AS neu, COUNT_IF(about_food) AS about_food
    FROM {TARGET}
""").collect()[0]
print(f"📊 {TARGET}: {s['n']} comments · positive {s['pos']} · negative {s['neg']} · neutral {s['neu']} · "
      f"about food {s['about_food']} · unparsed {s['unparsed']} " + ("✅" if s['n'] == s['grain'] else "❌ duplicates"))
assert s["n"] == s["grain"], "duplicate comment_id"